In [1]:
import pandas as pd

# rutas (ajusta según tu carpeta)
patients = pd.read_csv("./datasets/PATIENTS.csv")
admissions = pd.read_csv("./datasets/ADMISSIONS.csv")
icustays = pd.read_csv("./datasets/ICUSTAYS.csv")

In [2]:
# convertir fechas
patients["dob"] = pd.to_datetime(patients["dob"])
patients["dod"] = pd.to_datetime(patients["dod"])

admissions["admittime"] = pd.to_datetime(admissions["admittime"])
admissions["dischtime"] = pd.to_datetime(admissions["dischtime"])
admissions["deathtime"] = pd.to_datetime(admissions["deathtime"])

icustays["intime"] = pd.to_datetime(icustays["intime"])
icustays["outtime"] = pd.to_datetime(icustays["outtime"])

In [3]:
df = icustays.merge(admissions, on=["subject_id", "hadm_id"], how="left")
df = df.merge(patients, on="subject_id", how="left")

In [4]:
df["age"] = (df["intime"] - df["dob"]).dt.days / 365

# corregir edades irreales
df.loc[df["age"] > 100, "age"] = 90

In [5]:
df["icu_los_days"] = (df["outtime"] - df["intime"]).dt.total_seconds() / (60*60*24)

In [6]:
df["mortality"] = df["deathtime"].notnull().astype(int)

In [7]:
print(df.columns)

Index(['row_id_x', 'subject_id', 'hadm_id', 'icustay_id', 'dbsource',
       'first_careunit', 'last_careunit', 'first_wardid', 'last_wardid',
       'intime', 'outtime', 'los', 'row_id_y', 'admittime', 'dischtime',
       'deathtime', 'admission_type', 'admission_location',
       'discharge_location', 'insurance', 'language', 'religion',
       'marital_status', 'ethnicity', 'edregtime', 'edouttime', 'diagnosis',
       'hospital_expire_flag', 'has_chartevents_data', 'row_id', 'gender',
       'dob', 'dod', 'dod_hosp', 'dod_ssn', 'expire_flag', 'age',
       'icu_los_days', 'mortality'],
      dtype='str')


In [8]:
dataset = df[[
    "subject_id",
    "hadm_id",
    "icustay_id",
    "gender",
    "age",
    "icu_los_days",
    "mortality"
]]

In [9]:
dataset = dataset.dropna()
dataset = dataset[dataset["icu_los_days"] > 0]

In [10]:
print(dataset.head())
print(dataset.describe())
print(dataset.value_counts("mortality"))

   subject_id  hadm_id  icustay_id gender        age  icu_los_days  mortality
0       10006   142345      206504      F  70.682192      1.632546          0
1       10011   105331      232110      F  36.213699     13.850694          1
2       10013   165520      264446      F  87.142466      2.649907          1
3       10017   199207      204881      F  73.734247      2.143611          0
4       10019   177759      228977      M  48.931507      1.293843          1
         subject_id        hadm_id     icustay_id         age  icu_los_days  \
count    136.000000     136.000000     136.000000  136.000000    136.000000   
mean   28263.485294  153259.566176  250980.470588   70.871152      4.452461   
std    16008.281510   28054.220280   28455.125832   16.161731      6.196832   
min    10006.000000  100375.000000  201006.000000   17.202740      0.105926   
25%    10089.750000  129028.000000  224359.250000   63.877397      1.233504   
50%    40307.000000  157724.000000  250055.000000   74.723

In [11]:
dataset.to_csv("mimic_base_dataset.csv", index=False)